# Week 3 Tutorial: Pandas — Series and DataFrames

**AI in Data Science | Google Colab | Suggested time: 80–90 minutes**

## Driving question

How can we turn a messy table of student scores into trustworthy evidence about learning?

## Learning goals

By the end, you can create Series and DataFrames; load CSV and Excel data; inspect, select, filter, and sort data; find and repair missing values; correct data types; remove duplicates; and answer questions with evidence.

> Run cells in order. Before selected cells, pause and predict the result.

## 1. Pandas and the data-science workflow

**Pandas** is a Python library for table-shaped data. A **Series** is one labeled column. A **DataFrame** is a table made of multiple Series that share row labels.

Today’s workflow: **load → inspect → select → filter → sort → clean → analyze → communicate**.

In [ ]:
# Import pandas and use the standard short name pd.
import pandas as pd
# Print the installed Pandas version.
print("Pandas version:", pd.__version__)

## 2. Series: one labeled dimension

The values are the data; the labels on the left form the **index**.

In [ ]:
# Create a Series of game scores with player names as labels.
game_scores = pd.Series([120, 95, 140, 110], index=["Ava", "Ben", "Chloe", "Diego"], name="score")
# Display the complete Series.
print(game_scores)
# Select Chloe's score by its label.
print("Chloe's score:", game_scores.loc["Chloe"])
# Select the first score by its position.
print("First score:", game_scores.iloc[0])
# Calculate the average score.
print("Average score:", game_scores.mean())

### Checkpoint 1

1. What is the difference between a Series value and its index?
2. Predict the result of `game_scores.loc["Ben"]`.
3. Why is `.iloc[0]` safer and clearer than using `[0]` for position?

## 3. DataFrames: rows and columns

Each row is one student record. Each column is one variable.

In [ ]:
# Create a dictionary whose keys will become column names.
score_data = {"name": ["Ava", "Ben", "Chloe"], "math": [88, 91, 95], "science": [90, 89, 96]}
# Build a DataFrame from the dictionary.
score_table = pd.DataFrame(score_data)
# Display the DataFrame.
score_table

In [ ]:
# Select one column as a Series.
math_series = score_table["math"]
# Select two columns as a DataFrame.
two_columns = score_table[["name", "science"]]
# Print the Python type of the one-column selection.
print(type(math_series))
# Print the Python type of the two-column selection.
print(type(two_columns))
# Display the two selected columns.
two_columns

## 4. Load embedded CSV data

Real projects often begin with a CSV file. To keep this notebook self-contained, the CSV is stored as text and loaded from memory. It contains realistic quality problems for us to discover.

In [ ]:
# Import pandas and use the standard short name pd.
import pandas as pd
# Import StringIO so text in this notebook can behave like a CSV file.
from io import StringIO
# Store a small, intentionally messy student dataset as CSV text.
csv_text = """student_id,name,grade,math_pre,math_post,science_pre,science_post,study_hours,club
S01,Ava,9,78,88,82,90,4.5,Robotics
S02,Ben,10,85,91,80,89,5.0,AI Club
S03,Chloe,9,92,95,94,96,6.0,Science
S04,Diego,11,68,79,72,81,3.5,robotics
S05,Emma,10,88,,86,92,5.5,Art
S06,Finn,9,74,83,70,,4.0,AI Club
S07,Grace,11,90,94,91,95,6.5,Science
S08,Hassan,10,65,76,69,78,3.0, Robotics 
S09,Ivy,9,81,87,84,90,4.5,Art
S10,Jayden,11,77,85,75,84,five,AI Club
S10,Jayden,11,77,85,75,84,five,AI Club
S11,Kai,10,83,90,88,93,5.0,science
"""
# Read the embedded CSV text into a Pandas DataFrame.
students = pd.read_csv(StringIO(csv_text))
# Display the DataFrame.
students

### CSV and Excel

For an uploaded CSV, use `pd.read_csv("filename.csv")`. For Excel, use `pd.read_excel("filename.xlsx")`. In Colab, upload the file first using the Files panel. The next demonstration creates an Excel file in memory, so no upload is required.

In [ ]:
# Import BytesIO so memory can behave like a binary Excel file.
from io import BytesIO
# Create an empty in-memory binary file.
excel_file = BytesIO()
# Write the small score table to the in-memory Excel file.
score_table.to_excel(excel_file, index=False)
# Move the file pointer back to the beginning.
excel_file.seek(0)
# Read the in-memory Excel file into a new DataFrame.
excel_students = pd.read_excel(excel_file)
# Display the DataFrame loaded from Excel.
excel_students

## 5. Inspect before changing anything

Never clean blindly. First ask: How large is the table? What are the columns and data types? Where is data missing?

In [ ]:
# Display the first five rows.
display(students.head())
# Display the last three rows.
display(students.tail(3))
# Print the number of rows and columns.
print("Shape:", students.shape)
# Print all column names.
print("Columns:", students.columns.tolist())
# Print structural information, including data types and non-null counts.
students.info()

In [ ]:
# Display summary statistics for numeric columns.
display(students.describe())
# Count missing values in every column.
missing_counts = students.isna().sum()
# Display the missing-value counts.
display(missing_counts)

### Checkpoint 2

Find three data-quality problems. Hint: inspect the row count, missing-value counts, `study_hours` type, and spelling/spacing in `club`.

## 6. Select with brackets, `.loc`, and `.iloc`

- `df["column"]`: one column as a Series
- `df[["a", "b"]]`: multiple columns as a DataFrame
- `.loc`: labels or conditions
- `.iloc`: integer positions

In [ ]:
# Select the name column as a Series.
names = students["name"]
# Display the first five names.
display(names.head())
# Select three named columns as a DataFrame.
score_columns = students[["name", "math_pre", "math_post"]]
# Display the first five rows of the smaller DataFrame.
display(score_columns.head())
# Select rows 0 through 2 and two named columns by labels.
display(students.loc[0:2, ["name", "grade"]])
# Select the first three rows and first four columns by positions.
display(students.iloc[0:3, 0:4])

## 7. Filter rows with questions

Each comparison creates a Boolean Series. Use `&` for AND, `|` for OR, and parentheses around each condition.

In [ ]:
# Create a Boolean condition for students whose math pre-score is below 80.
needs_support_mask = students["math_pre"] < 80
# Display the first five True or False values.
display(needs_support_mask.head())
# Keep matching rows and show only useful columns.
needs_support = students.loc[needs_support_mask, ["name", "math_pre", "math_post"]]
# Display the filtered results.
display(needs_support)
# Keep grade 9 students whose science pre-score is at least 80.
grade9_science = students.loc[(students["grade"] == 9) & (students["science_pre"] >= 80), ["name", "science_pre"]]
# Display the results of the two-condition filter.
display(grade9_science)

## 8. Sort to reveal rank and order

In [ ]:
# Sort students from highest to lowest math post-score.
math_ranking = students.sort_values(by="math_post", ascending=False)
# Display the top five names and math post-scores.
display(math_ranking[["name", "math_post"]].head())
# Sort by grade first and math post-score second.
grade_ranking = students.sort_values(by=["grade", "math_post"], ascending=[True, False])
# Display selected columns from the multi-column sort.
display(grade_ranking[["name", "grade", "math_post"]])

## 9. Clean the dataset safely

We will work on a copy, normalize text, remove duplicates, convert a data type, and fill missing post-scores with the median. In a real study, the missing-value strategy must be justified and documented.

In [ ]:
# Make a copy so the original raw data remains unchanged.
clean_students = students.copy()
# Remove leading and trailing spaces from club names.
clean_students["club"] = clean_students["club"].str.strip()
# Convert club names to consistent title capitalization.
clean_students["club"] = clean_students["club"].str.title()
# Convert study hours to numbers and turn invalid text into NaN.
clean_students["study_hours"] = pd.to_numeric(clean_students["study_hours"], errors="coerce")
# Remove completely duplicated rows.
clean_students = clean_students.drop_duplicates()
# Calculate the median math post-score.
math_post_median = clean_students["math_post"].median()
# Fill missing math post-scores with the median.
clean_students["math_post"] = clean_students["math_post"].fillna(math_post_median)
# Calculate the median science post-score.
science_post_median = clean_students["science_post"].median()
# Fill missing science post-scores with the median.
clean_students["science_post"] = clean_students["science_post"].fillna(science_post_median)
# Calculate the median number of study hours.
study_hours_median = clean_students["study_hours"].median()
# Fill missing study hours with the median.
clean_students["study_hours"] = clean_students["study_hours"].fillna(study_hours_median)
# Reset row labels after removing a duplicate.
clean_students = clean_students.reset_index(drop=True)
# Display the cleaned table.
clean_students

In [ ]:
# Print the cleaned table's shape.
print("Clean shape:", clean_students.shape)
# Display the cleaned data types.
display(clean_students.dtypes)
# Display the remaining missing-value counts.
display(clean_students.isna().sum())
# Display each standardized club name and its frequency.
display(clean_students["club"].value_counts())

### Checkpoint 3

Why did we keep `students` unchanged and clean a copy? What are the benefits and risks of median imputation?

## 10. Analyze student improvement

Creating improvement columns is a simple transformation. Formal feature engineering and grouped analysis arrive in Week 4.

In [ ]:
# Calculate each student's math improvement.
clean_students["math_improvement"] = clean_students["math_post"] - clean_students["math_pre"]
# Calculate each student's science improvement.
clean_students["science_improvement"] = clean_students["science_post"] - clean_students["science_pre"]
# Calculate each student's average improvement across both subjects.
clean_students["average_improvement"] = clean_students[["math_improvement", "science_improvement"]].mean(axis=1)
# Sort students from greatest to smallest average improvement.
improvement_ranking = clean_students.sort_values(by="average_improvement", ascending=False)
# Display the improvement evidence.
display(improvement_ranking[["name", "math_improvement", "science_improvement", "average_improvement"]])

In [ ]:
# Calculate the average math post-score.
average_math_post = clean_students["math_post"].mean()
# Calculate the average science post-score.
average_science_post = clean_students["science_post"].mean()
# Find the row label of the largest average improvement.
best_index = clean_students["average_improvement"].idxmax()
# Select the name at that row label.
most_improved_name = clean_students.loc[best_index, "name"]
# Select the improvement value at that row label.
most_improved_value = clean_students.loc[best_index, "average_improvement"]
# Print the subject averages with one decimal place.
print(f"Average math post-score: {average_math_post:.1f}")
# Print the science average with one decimal place.
print(f"Average science post-score: {average_science_post:.1f}")
# Print the most improved student and the supporting value.
print(f"Most improved student: {most_improved_name} ({most_improved_value:.1f} points)")

## 11. Communicate findings responsibly

A strong conclusion contains a **claim**, **numerical evidence**, and a **limitation**.

Example structure: “Science had the higher post-test average (___ versus ___). ___ improved the most by an average of ___ points. However, the dataset is small and missing values were replaced with medians, so the result should not be generalized.”

### Exit ticket

1. When would you use a Series instead of a DataFrame?
2. Explain `.loc` versus `.iloc` in one sentence.
3. Name one cleaning decision that could influence a conclusion.
4. Write one claim, one piece of evidence, and one limitation from today’s data.

## Skills checklist

- [ ] I can explain Series and DataFrames.
- [ ] I can load CSV and Excel data.
- [ ] I inspect data before cleaning it.
- [ ] I can select, filter, and sort.
- [ ] I can identify missing values and incorrect types.
- [ ] I can clean without destroying the raw data.
- [ ] I can support a conclusion with numbers and limitations.

**Next:** complete the guided exercise notebook.